## Opgave 1

Zoek de rij met de hoogste som in een matrix. Loop daarvoor door de rijen en controleer voor elke rij elk element van links naar rechts. Gebruik het bewijs van het tegendeel om te stoppen als de huidige rij al een strikt hogere som heeft dan de huidige beste oplossing. Als resultaat geef je de index (van 0 te tellen) van de beste rij `beste_rij`, de som van die rij `hoogste_som` en een dictionary `controles` die bijhoudt voor elke rij hoeveel elementen je hebt moeten controleren voor je (g)een tegenvoorbeeld vond (want als je een tegenvoorbeeld vindt moet je niet tot het eind van de rij lopen).

Voorbeeldapplicatie: vind het aandeel met de hoogste return, vind de machine met de hoogste productie, enz.

In [ ]:
def vind_hoogste_som(matrix) -> (int, int, dict):
    
    # variabelen waarin de antwoorden
    # worden bijgehouden
    controles = {}
    rij = []
    hoogste_som = 0
    beste_rij = 0
    
    # loop door de rijen van de matrix
    for r, rij in enumerate(matrix):
        print('Controleer rij', rij)
        # initialiseer de som
        som = 0
        # loop door de elementen van de rij
        for e, element in enumerate(rij):
            # verhoog de som
            som += element
            # controleer of de som al hoger is
            if som > hoogste_som:
                print('Betere oplossing')
                hoogste_som = sum(rij)
                beste_rij = r
                break
        # vul 'controles' in
        controles[r] = e
    
    # niet aanpassen
    return beste_rij, hoogste_som, controles

In [2]:
matrix = [[10, 2, 3, 4, 4], [10 ,2 ,2], [8, 8888, 8], [1, 10,1 ,1 ,1], [0, 15, -56, 20000]]

beste_rij, hoogste_som, controles = vind_hoogste_som(matrix)
print('\nBeste rij:', beste_rij)
print('Hoogste som:', hoogste_som)
print('Controles:', controles)

assert beste_rij == 4
assert hoogste_som == 19959
assert controles == {0: 0, 1: 2, 2: 1, 3: 4, 4: 3}

Controleer rij [10, 2, 3, 4, 4]
Betere oplossing
Controleer rij [10, 2, 2]
Controleer rij [8, 8888, 8]
Betere oplossing
Controleer rij [1, 10, 1, 1, 1]
Controleer rij [0, 15, -56, 20000]
Betere oplossing

Beste rij: 4
Hoogste som: 19959
Controles: {0: 0, 1: 2, 2: 1, 3: 4, 4: 3}


In [3]:
matrix = [[10, 2, 3, 4, 4], [10 ,2 ,2], [8, 2, 8], [1, 10,1 ,1 ,1], [0, 15, -56, 10]]

beste_rij, hoogste_som, controles = vind_hoogste_som(matrix)
print('\nBeste rij:', beste_rij)
print('Hoogste som:', hoogste_som)
print('Controles:', controles)

assert beste_rij == 0
assert hoogste_som == 23
assert controles == {0: 0, 1: 2, 2: 2, 3: 4, 4: 3}

Controleer rij [10, 2, 3, 4, 4]
Betere oplossing
Controleer rij [10, 2, 2]
Controleer rij [8, 2, 8]
Controleer rij [1, 10, 1, 1, 1]
Controleer rij [0, 15, -56, 10]

Beste rij: 0
Hoogste som: 23
Controles: {0: 0, 1: 2, 2: 2, 3: 4, 4: 3}


## Opgave 2

Gegeven: de verkorte code van 2A.0 (24-urenloop) zonder de visualisatie en het wegschrijven naar een dataframe.

Voeg de volgende 2 elementen toe aan de code (de plaatsen waar dit moet gebeuren zijn aangegeven):

- Voeg 1 team toe 'Pedal', die een gemiddelde van 140 hebben en een standaarddeviatie van 15 in de normaalverdeling. (1 punt)

- Voeg aan de simulatie vermoeiheid toe (2 punten). Zorg ervoor dat, indien de laatste 5 rondes sneller waren dan gemiddeld overheen heel de historiek er een penalty van 20% wordt aangerekend van het verschil tussen het volledig gemiddelde en het gemiddelde over de laatste 5 rondes. Deze penalty wordt teruggegeven als resultaat en kan bij de rondetijd geteld worden waar de functie wordt opgeroepen. Indien het gemiddelde lager is, geef 0 terug.

Voorbeeld 1: gemiddelde gehele historiek = 10, gemiddelde laatste 5 = 8, resultaat = 0.2 * (10-8).

Voorbeeld 2: gemiddelde gehele historiek = 8, gemiddelde laatste 5 = 9, resultaat = 0.

TIP: als je je code nakijkt gebruik je best de debugger, aangezien er veel rondes worden berekend en extra print-statements zullen zorgen voor heel veel output.

In [ ]:
import numpy as np
import pandas as pd
from typing import Callable, Dict, Tuple

np.random.seed(10)

SECONDEN_PER_UUR = 3600
RACE_SECONDEN = 24 * SECONDEN_PER_UUR


def trek_rondetijd(verdeling: str, gemiddelde: float, sd: float, aantal: int=1, rng: np.random.Generator=None):
    """Geef een steekproef (of meerdere) van rondetijden (in seconden) uit de gevraagde verdeling.
    gemiddelde en sd zijn het gemiddelde en de standaardafwijking van de rondetijd,
    ook bij verdelingen die intern andere parameters gebruiken (die rekenen we om).
    Ondersteunde verdelingen: 'deterministisch', 'normaal', 'exponentieel'
    """
    if rng is None:
        rng = np.random.default_rng()

    if verdeling == 'normaal':
        # Begrens naar onder op een kleine positieve waarde om niet-positieve tijden te vermijden
        steekproef = rng.normal(loc=gemiddelde, scale=sd, size=aantal)
        steekproef = np.clip(steekproef, 0.1, None)
        return steekproef

    if verdeling == 'exponentieel':
        # Exponentieel met gemiddelde = 1/lambda
        lam = 1.0 / gemiddelde
        return rng.exponential(scale=1.0/lam, size=aantal)

    raise ValueError(f"Onbekende verdeling: {verdeling}")


def ronden_in_tijd(rondetijden: np.ndarray, totale_tijd: float) -> int:
    """Tel hoeveel volledige ronden passen in totale_tijd seconden gegeven opeenvolgende rondetijden."""
    cumulatief = np.cumsum(rondetijden)
    # aantal ronden waarvan de cumulatieve tijd <= totale_tijd
    return int(np.searchsorted(cumulatief, totale_tijd, side='right'))

def voeg_vermoeidheid_toe(historiek: list[float]):
    resultaat = 0

    # zorg ervoor dat, indien de laatste 5 rondes sneller waren dan gemiddeld overheen heel de historiek
    # er een penalty van 20% wordt aangerekend van het verschil tussen het volledig gemiddelde en het gemiddelde over
    # de laatste 5 rondes
    # deze penalty wordt teruggegeven als resultaat en kan bij de rondetijd geteld worden waar de functie
    # wordt opgeroepen

    laatste_5 = historiek[-5:]
    gemiddelde_laatste_5 = np.mean(laatste_5)

    # Faster recent laps -> more fatigue
    inspanning = max(0, np.mean(historiek) - gemiddelde_laatste_5)

    resultaat = 0.2 * inspanning

    return resultaat


def simuleer_race(team_parameters: Dict[str, Dict], rng: np.random.Generator) -> Dict[str, int]:
    """Simuleer één race: genereer per team rondetijden tot 24 uur en tel ronden.
    team_parameters: {teamnaam: {'verdeling': str, 'gemiddelde': float, 'sd': float}}
    geeft terug: {teamnaam: afgewerkte_ronden}
    """
    resultaten = {}
    for team, p in team_parameters.items():
        # We genereren de rondetijden één voor één tot we RACE_SECONDEN overschrijden.
        ronden = 0
        ronde_tijden = []
        bestede_tijd = 0.0
        # veiligheidsgrens: genereer geen miljarden steekproeven
        while bestede_tijd < RACE_SECONDEN:
            rondetijd = trek_rondetijd(p.get('verdeling','normaal'), p['gemiddelde'], p.get('sd', 0.0), aantal=1, rng=rng)[0]
            
            ######################
            # Deze functie is nieuw 
            # hier wordt de penalty toegevoegd
            if len(ronde_tijden) >= 5:
                rondetijd = rondetijd + voeg_vermoeidheid_toe(ronde_tijden)
            ######################

            ronde_tijden.append(rondetijd)

            if bestede_tijd + rondetijd > RACE_SECONDEN:
                break
            bestede_tijd += rondetijd
            ronden += 1

            # optionele vroegtijdige stop als het aantal ronden absurd groot wordt (veiligheid)
            if ronden > 1000000:
                break
        resultaten[team] = ronden
    return resultaten


def voer_simulatie_uit(team_parameters: Dict[str, Dict], aantal_simulaties: int = 20000, zaad: int = 42, uitgebreid: bool = False) -> pd.DataFrame:
    rng = np.random.default_rng(zaad)
    records = []
    for t in range(aantal_simulaties):
        race = simuleer_race(team_parameters, rng)
        # noteer de winnaar (bij gelijke stand: de winst wordt verdeeld over de teams met evenveel ronden)
        max_ronden = max(race.values())
        winnaars = [team for team, ronden in race.items() if ronden == max_ronden]
        for team, ronden in race.items():
            records.append({
                'simulatie': t,
                'team': team,
                'ronden': ronden,
                'winaandeel': 1.0/len(winnaars) if team in winnaars else 0.0
            })
        if uitgebreid and (t+1) % (aantal_simulaties//5) == 0:
            print(f"{t+1}/{aantal_simulaties} simulaties klaar")
    df = pd.DataFrame.from_records(records)
    return df


def vat_resultaten_samen(df: pd.DataFrame) -> pd.DataFrame:
    # winkans = som(winaandeel)/aantal_simulaties
    aantal_simulaties = df['simulatie'].nunique()
    winkansen = df.groupby('team')['winaandeel'].sum().rename('winkans') / 1.0
    gem_ronden = df.groupby('team')['ronden'].mean().rename('gem_ronden')
    std_ronden = df.groupby('team')['ronden'].std().rename('std_ronden')
    samenvatting = pd.concat([winkansen, gem_ronden, std_ronden], axis=1)
    samenvatting['winkans'] = samenvatting['winkans'] / aantal_simulaties
    samenvatting = samenvatting.sort_values('winkans', ascending=False)

    return samenvatting

# Team-parameters (rondetijd in seconden):

##################
# VUL hieronder 'Pedal' aan met de gevraagde parameters. 
##################

teams_met_parameters = {
    'Ekonomika': {'verdeling': 'normaal', 'gemiddelde': 81.0, 'sd': 10.0},
    # 'VTK':       {'verdeling': 'normaal', 'gemiddelde': 78.0, 'sd': 12.0},
    # 'Apolloon':  {'verdeling': 'exponentieel', 'gemiddelde': 77.0, 'sd': 40.0},
    # 'Pedal':    {'verdeling': 'normaal', 'gemiddelde': 140, 'sd': 15}
}

print(f"Simulatie wordt uitgevoerd:")
df = voer_simulatie_uit(teams_met_parameters, aantal_simulaties=50, zaad=12345)
samenvatting = vat_resultaten_samen(df)
print(samenvatting)

print("Klaar. Resultaten zijn opgeslagen in resultaten_<strategie>/ indien niet in commentaar gezet.")

Simulatie wordt uitgevoerd:
0.0
0.0
0.33298447862386243
0.006460817792554963
0.0
0.33047402275232685
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.029868455954002116
0.0
0.0
0.0
0.0
0.0
0.0
0.3436294272749166
0.6485090214980176
1.0168564456955125
0.0
0.0
0.0
0.0
0.0
0.0
0.8944564316339552
0.9851539283791682
1.018700232500413
0.8238401762324288
0.7360993825449583
0.05321165542160316
0.19147664596105188
0.5683843900679421
0.0
0.0
0.0
0.11677057037329348
0.017293213189233825
0.0626129589149258
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.5197810168479094
0.6153454063859812
1.3001994516355297
0.9588386986861736
0.8675086376077957
0.34526309669608396
0.42224417523451624
0.5305362407661874
0.8503725469874042
0.7915197964885409
1.0578958681490207
0.8297958224112307
0.6746524530640159
0.665101680288808
1.3575719324199866
0.8747196423352706
0.945767080891227
0.7864482125109902
0.4894812774197874
0.0
0.0
0.18408419083507113
0.0
0.0
0.6193806537685135
1.1446627006093735
0.46842107666061567
0.8284656340842276
0.7083